# Alzheimer's Triage Agent - Demo

This notebook demonstrates the Alzheimer's Triage Agent. It follows the structured 1-question-at-a-time flow.

In [1]:
import os
import json
from alzheimer_agent import AlzheimerAgent
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
if not os.getenv("GOOGLE_API_KEY"):
    print("WARNING: GOOGLE_API_KEY not found.")

# Debug: List available models
try:
    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    print("Available models:")
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(m.name)
except Exception as e:
    print(f"Error listing models: {e}")

Available models:
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
model

In [2]:
agent = AlzheimerAgent(session_id="alz_demo_01")
print("Alzheimer Agent initialized.")

Alzheimer Agent initialized.


## Simulation Loop

In [3]:
def run_turn(user_input):
    print(f"\nUser: {user_input}")
    response = agent.step(user_input)
    print(f"Agent ({response['type']}):")
    print(json.dumps(response, indent=2))
    return response

# Turn 1: Initial Complaint
resp1 = run_turn("I keep forgetting recent conversations and it's getting worrying.")

# Turn 2: Answer Follow-up (Duration)
if resp1['type'] == 'followup':
    resp2 = run_turn("It's been happening for about 6 months now.")

# Turn 3: Answer Follow-up (Daily Tasks)
if 'resp2' in locals() and resp2['type'] == 'followup':
    resp3 = run_turn("Yes, I sometimes struggle to manage my bills.")

# Turn 4: Answer Follow-up (Family History)
if 'resp3' in locals() and resp3['type'] == 'followup':
    resp4 = run_turn("My mother had dementia.")

# Turn 5: Answer Follow-up (Confusion)
if 'resp4' in locals() and resp4['type'] == 'followup':
    resp5 = run_turn("I got lost driving home once last week.")


User: I keep forgetting recent conversations and it's getting worrying.


2025-11-28 17:19:27,759 - ERROR - Feature extraction error: 'list' object has no attribute 'items'
2025-11-28 17:19:28,560 - INFO - Scores - Rule: 0.05, LLM: 0.00, Final: 0.02


Agent (followup):
{
  "type": "followup",
  "question": "Do you forget recent conversations often?",
  "feature_target": "forget_recent",
  "expected_answers": [
    "yes",
    "no",
    "sometimes"
  ],
  "round": 1,
  "session_id": "alz_demo_01"
}

User: It's been happening for about 6 months now.


2025-11-28 17:19:30,187 - INFO - Scores - Rule: 0.05, LLM: 0.00, Final: 0.02


Agent (followup):
{
  "type": "followup",
  "question": "Are you having trouble with daily tasks?",
  "feature_target": "daily_task_difficulty",
  "expected_answers": [
    "yes",
    "no"
  ],
  "round": 2,
  "session_id": "alz_demo_01"
}

User: Yes, I sometimes struggle to manage my bills.


...

Agent (followup):
{
  "type": "followup",
  "question": "Do you feel confused about dates or places?",
  "feature_target": "confused_time_place",
  "expected_answers": [
    "yes",
    "no"
  ],
  "round": 3,
  "session_id": "alz_demo_01"
}

User: My mother had dementia.


2025-11-28 17:19:33,152 - ERROR - Feature extraction error: 'list' object has no attribute 'items'
2025-11-28 17:19:34,078 - INFO - Scores - Rule: 0.05, LLM: 0.00, Final: 0.02


Agent (followup):
{
  "type": "followup",
  "question": "Any family history of dementia?",
  "feature_target": "family_history",
  "expected_answers": [
    "yes",
    "no"
  ],
  "round": 4,
  "session_id": "alz_demo_01"
}

User: I got lost driving home once last week.


2025-11-28 17:19:36,431 - ERROR - Feature extraction error: 'list' object has no attribute 'items'
2025-11-28 17:19:37,363 - INFO - Scores - Rule: 0.05, LLM: 0.00, Final: 0.02


Agent (followup):
{
  "type": "followup",
  "question": "How many hours do you sleep nightly?",
  "feature_target": "sleep_hours",
  "expected_answers": [
    "numeric"
  ],
  "round": 5,
  "session_id": "alz_demo_01"
}


## Check Internal State

In [4]:
print("Features:")
print(json.dumps(agent.session.features, indent=2))
print("\nNormalized:")
print(json.dumps(agent.session.normalized_features, indent=2))

Features:
{
  "duration_months": "6",
  "forget_recent": "no",
  "daily_task_difficulty": "error_skipped",
  "confused_time_place": "error_skipped",
  "family_history": "error_skipped"
}

Normalized:
{
  "duration_months": 0.25,
  "forget_recent": 0.0,
  "daily_task_difficulty": 0.0,
  "confused_time_place": 0.0,
  "family_history": 0.0
}
